# **Seminar 5 — Federated Learning**

1. **Part I: ML model definition**
    
    - The dataset is known to you, so you are already able to load and process it as needed.
    
    - Focus on the ML model that you will train collaboratively through FL.
    
    - You are free to choose the model architecture (you can also work with the linear regressor from Seminar 2).
    
    - Remember that we are dealing with a classification problem, which may have implications in your model architecture (e.g., selected loss function, type of activations).

2. **Part II: Preparation of the FL setting**
    - Define a central orchestrator (or parameter server) which must perform the following operations:
        
        1. Initialize a global ML model, $ω(t = 0)$
        
        2. Select a subset of clients S ⊆ K to participate in the current training iteration.
        
        3. Send the global model to the clients, which is retrained by the clients using their local datasets $(D^k, ∀k ∈ S)$.
        
        4. Retrieve the individual models $ω_k(t)$, $∀k ∈ S$ from the selected clients.
        
        5. Aggregate the individual contributions to update the global model, $ω(t+1)$. To perform model aggregation, consider using FedAvg, whereby a weighted average of the model weights is calculated. The weights ($α$) are given by the total share of data of each client: $\alpha_k = \frac{|D^k|}{\sum_{k \in S} |D^k|}, \quad k \in S$
        
        6. Repeat steps 2-5 until convergence.

3. **Part III: Collaborative training of the model**
    
    - Define the required hyperparameters to collaboratively train your ML model (e.g., number of FL iterations, number of epochs for local model training, batch size, etc.).
    
    - Train the model using FL using the training dataset (client_k_features.csv and client_k_labels.csv).
    
    - Evaluate the model using the test data partition (test_features.csv and test_labels.csv).

#### **Core libraries**

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report

# **Part I**

#### **1. Model Architecture Definition**

We implement a **multiclass Logistic Regression** model using pure NumPy. This choice was made as we need to manually transfer model weights between the server and clients during FL, so we need full control over the weight matrices (`coef` and `intercept`). 

**The architecture is as it follows:**
- **Input:** 270 features (flattened Wi-Fi CSI matrix: 30 × 3 × 3)
- **Output:** 12 classes (one per pose)
- **Activation:** Softmax (converts raw scores to probabilities)
- **Loss:** Cross-entropy (standard for multiclass classification)
- **Optimizer:** Gradient Descent

In [3]:
# ── Model constants ──────────────────────────────────────────────────────────
N_CLASSES  = 12   # poses: wave, push, crouch, sitdown, bend, ...
N_FEATURES = 270  # flattened CSI matrix: 30×3×3

def softmax(z):
    """
    Numerically stable softmax.
    Input z: (n_samples, n_classes)
    Output: probability matrix of same shape.
    """
    e = np.exp(z - z.max(axis=1, keepdims=True))  # subtract max for stability
    return e / e.sum(axis=1, keepdims=True)

def predict(X, coef, intercept):
    """
    Predict class labels (1–12) for input X.
    coef:      (12, 270)
    intercept: (12,)
    """
    logits = X @ coef.T + intercept    # (n_samples, 12)
    probs  = softmax(logits)           # (n_samples, 12)
    return np.argmax(probs, axis=1) + 1  # classes are 1-indexed

print("Model functions defined.")
print(f"Weight matrix shape: ({N_CLASSES}, {N_FEATURES})")
print(f"Bias vector shape:   ({N_CLASSES},)")

Model functions defined.
Weight matrix shape: (12, 270)
Bias vector shape:   (12,)


**Explanation of how logistic regression works here:** For each sample, the model computes a score for each of the 12 classes (a linear combination of the 270 features). Softmax turns these scores into probabilities. The class with the highest probability is the prediction. During training, gradient descent updates the weights to reduce the cross-entropy loss.

# **Part II**

#### **1. Central Orchestrator (Parameter Server)**

The parameter server is the coordinator of the FL process. It never sees any raw data, it only works with model weights. 

Therefore, its responsibilities are:
1. Hold and initialize the global model weights
2. Select which clients participate in each round
3. Distribute the global weights to selected clients
4. Collect the locally-trained weights from clients
5. Aggregate them using **FedAvg**: a weighted average proportional to each client's dataset size

In [4]:
class ParameterServer:

    def __init__(self, n_features, n_classes, random_state=42):
        self.random_state = random_state
        random.seed(random_state)
        
        # we initialize global model weights to zero
        self.coef = np.zeros((n_classes, n_features))  # (12, 270)
        self.intercept = np.zeros(n_classes) # (12,)
        print(f"Global model initialized — coef: {self.coef.shape}, intercept: {self.intercept.shape}")

    def get_global_weights(self):
        # we return a copy of the current global weights to send to clients 
        return self.coef.copy(), self.intercept.copy()

    def select_clients(self, all_client_ids, fraction=1.0):
        # we randomly select a subset S ⊆ K of clients for this round
        k = max(1, int(len(all_client_ids) * fraction))
        return random.sample(all_client_ids, k)

    def aggregate(self, client_updates, client_sizes):
        # now the FedAvg aggregation, each client's contribution is weighted by its share of total data
        # alpha_k = |D_k| / sum(|D_k|)    for k in selected clients
        # w(t+1)  = sum_k alpha_k * w_k(t)
        
        total_samples = sum(client_sizes)
        new_coef = np.zeros_like(self.coef)
        new_intercept = np.zeros_like(self.intercept)

        for (coef_k, intercept_k), n_k in zip(client_updates, client_sizes):
            alpha_k = n_k / total_samples  # weight for this client
            new_coef += alpha_k * coef_k
            new_intercept += alpha_k * intercept_k

        self.coef = new_coef
        self.intercept = new_intercept

print("ParameterServer class defined.")

ParameterServer class defined.


#### **2. Client Implementation**

Each client represents a device with its own local dataset. Clients:
- Receive the global model weights from the server
- Train for a few local epochs on their own data
- Send the updated weights back to the server

Importantly, clients **never share their raw data** as they only weight updates.

In [5]:
class Client:

    def __init__(self, client_id, X, y, learning_rate=0.1):
        self.client_id = client_id
        self.learning_rate = learning_rate
        
        # Normalize features locally (each client scales its own data)
        self.scaler = StandardScaler()
        self.X = self.scaler.fit_transform(X)
        self.y = y
        self.n = len(y)  # number of local samples — used for FedAvg weights

    def local_train(self, global_coef, global_intercept, local_epochs):
        # Train locally for `local_epochs` starting from the global model weights.
        # Start from the global model (warm start)
        
        coef = global_coef.copy()
        intercept = global_intercept.copy()

        # One-hot encode labels
        Y_onehot = np.zeros((self.n, N_CLASSES))
        for i, label in enumerate(self.y):
            Y_onehot[i, label - 1] = 1  # classes are 1-indexed

        for _ in range(local_epochs):
            # Forward pass: compute probabilities
            logits = self.X @ coef.T + intercept  # (n, 12)
            probs = softmax(logits) # (n, 12)

            # Compute gradients of cross-entropy loss
            diff = probs - Y_onehot # (n, 12)
            grad_coef = (diff.T @ self.X) / self.n # (12, 270)
            grad_interc = diff.mean(axis=0) # (12,)

            # Gradient descent update
            coef -= self.learning_rate * grad_coef
            intercept -= self.learning_rate * grad_interc

        return coef, intercept

print("Client class defined.")

Client class defined.


**Explanation, why warm-start?:** Each client begins local training from the global model weights (not from scratch). This is the key idea of FedAvg: the global model captures knowledge from all previous rounds, and each client fine-tunes it on their local data.

# **Part III**

#### **1. Hyperparameter Definition**

In [6]:
NUM_CLIENTS = 10     # total number of clients (K)
NUM_ROUNDS = 100    # number of FL communication rounds (T)
CLIENT_FRACTION = 1.0    # fraction of clients selected each round (1.0 = all)
LOCAL_EPOCHS = 10     # number of gradient descent steps per client per round
LEARNING_RATE = 0.1    # step size for local gradient descent
RANDOM_STATE = 42     # for reproducibility

print("Hyperparameters set:")
print(f"  Clients per round : {int(NUM_CLIENTS * CLIENT_FRACTION)} / {NUM_CLIENTS}")
print(f"  FL rounds         : {NUM_ROUNDS}")
print(f"  Local epochs      : {LOCAL_EPOCHS}")
print(f"  Learning rate     : {LEARNING_RATE}")

Hyperparameters set:
  Clients per round : 10 / 10
  FL rounds         : 100
  Local epochs      : 10
  Learning rate     : 0.1


#### **2. Data Loading**

We load each client's local dataset and the shared test set. Note that the label CSVs are stored as a single row (wide format), so we transpose them to get a column vector.

In [9]:
BASE_PATH = "." 

def load_client_data(client_id, base_path=BASE_PATH):
    # Load features and labels for a given client 
    X = pd.read_csv(f"{base_path}/client_datasets/client_{client_id}_features.csv",header=None).values
    y = pd.read_csv(f"{base_path}/client_datasets/client_{client_id}_labels.csv",header=None).T.values.ravel()
    return X, y

def load_test_data(base_path=BASE_PATH):
    # Load the shared test dataset
    X = pd.read_csv(f"{base_path}/test_features.csv", header=None).values
    y = pd.read_csv(f"{base_path}/test_labels.csv",   header=None).T.values.ravel()
    return X, y

# Instantiate clients
clients = {}
print(f"{'Client':<10} {'Samples':<10} {'Classes'}")
print("-" * 45)
for i in range(1, NUM_CLIENTS + 1):
    X, y = load_client_data(i)
    clients[i] = Client(i, X, y, learning_rate=LEARNING_RATE)
    print(f"Client {i:<4}  {len(y):<10} {sorted(set(y))}")

# Load and scale test data
X_test, y_test = load_test_data()
scaler_test = StandardScaler()
X_test_scaled = scaler_test.fit_transform(X_test)
print(f"\nTest set: {len(y_test)} samples, classes: {sorted(set(y_test))}")

Client     Samples    Classes
---------------------------------------------
Client 1     314        [np.int64(1), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(9), np.int64(11)]
Client 2     113        [np.int64(2), np.int64(7)]
Client 3     365        [np.int64(2), np.int64(4), np.int64(8), np.int64(9), np.int64(10), np.int64(11)]
Client 4     207        [np.int64(1), np.int64(6), np.int64(9)]
Client 5     209        [np.int64(3), np.int64(6), np.int64(8)]
Client 6     202        [np.int64(5), np.int64(6), np.int64(12)]
Client 7     448        [np.int64(1), np.int64(4), np.int64(5), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12)]
Client 8     142        [np.int64(5), np.int64(6), np.int64(9), np.int64(11)]
Client 9     412        [np.int64(1), np.int64(2), np.int64(6), np.int64(8), np.int64(9), np.int64(11), np.int64(12)]
Client 10    64         [np.int64(2), np.int64(5), np.int64(7)]

Test set: 500 samples, classes: [np.int64(1), np.int64(2), np.int

####  **3. Federated Learning Training**

#### **4. Model Evaluation**